In [ ]:
%%bash
cat > main.cpp <<'CD'
#include <bits/stdc++.h>
using namespace std;

struct PushRelabel {
    struct Aresta {
        int para;
        int rev;
        long long cap;
    };

    int vertices;
    vector<vector<Aresta>> adj;
    vector<long long> excesso;
    vector<int> altura;
    vector<int> prox;

    PushRelabel(int vertices_=0){ init(vertices_); }

    void init(int vertices_) {
        vertices = vertices_;
        adj.assign(vertices, {});
        excesso.assign(vertices, 0);
        altura.assign(vertices, 0);
        prox.assign(vertices, 0);
    }

    void adicionarAresta(int u, int v, long long capacidade) {
        Aresta direta{v, (int)adj[v].size(), capacidade};
        Aresta reversa{u, (int)adj[u].size(), 0};
        adj[u].push_back(direta);
        adj[v].push_back(reversa);
    }

    void empurrar(int u, Aresta &e) {
        if (excesso[u] == 0) return;
        int v = e.para;
        if (e.cap == 0) return;
        if (altura[u] != altura[v] + 1) return;

        long long envia = min(excesso[u], e.cap);

        e.cap -= envia;
        adj[v][e.rev].cap += envia;

        excesso[u] -= envia;
        excesso[v] += envia;
    }

    void rotular(int u) {
        int melhor = INT_MAX;
        for (auto &e : adj[u]) if (e.cap > 0) melhor = min(melhor, altura[e.para]);
        if (melhor < INT_MAX) altura[u] = melhor + 1;
    }

    void descarregar(int u) {
        while (excesso[u] > 0) {
            if (prox[u] == (int)adj[u].size()) {
                rotular(u);
                prox[u] = 0;
                continue;
            }
            Aresta &e = adj[u][prox[u]];
            if (e.cap > 0 && altura[u] == altura[e.para] + 1) empurrar(u, e);
            else prox[u]++;
        }
    }

    long long fluxoMaximo(int fonte, int destino) {
        altura.assign(vertices, 0);
        excesso.assign(vertices, 0);
        prox.assign(vertices, 0);

        altura[fonte] = vertices;

        for (auto &e : adj[fonte]) {
            if (e.cap > 0) {
                long long envia = e.cap;
                e.cap = 0;
                adj[e.para][e.rev].cap += envia;
                excesso[e.para] += envia;
                excesso[fonte] -= envia;
            }
        }

        list<int> ativos;
        for (int v = 0; v < vertices; v++) if (v != fonte && v != destino) ativos.push_back(v);

        auto it = ativos.begin();
        while (it != ativos.end()) {
            int u = *it;
            int altura_antiga = altura[u];
            descarregar(u);
            if (altura[u] > altura_antiga) {
                ativos.erase(it);
                ativos.push_front(u);
                it = ativos.begin();
            } else {
                ++it;
            }
        }

        return excesso[destino];
    }
};

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int vertices, arestas, fonte, destino;
    if (!(cin >> vertices >> arestas >> fonte >> destino)) return 0;

    PushRelabel pr(vertices);

    for (int i = 0; i < arestas; i++) {
        int u, v;
        long long capacidade;
        cin >> u >> v >> capacidade;
        pr.adicionarAresta(u, v, capacidade);
    }

    cout << pr.fluxoMaximo(fonte, destino) << "\n";
    return 0;
}
CD

g++ -std=c++17 -O2 main.cpp -o maxflow

cat > input.txt <<'EN'
6 10 0 5
0 1 16
0 2 13
1 2 10
2 1 4
1 3 12
2 4 14
3 2 9
3 5 20
4 3 7
4 5 4
EN

./maxflow < input.txt


23
